<a href="https://colab.research.google.com/github/aiwithpavansama/GenerativeandAgenticAI/blob/main/Automating_the_Resume_Screening_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install -q sentence-transformers pymupdf python-docx scikit-learn pandas

In [41]:
import os, re, glob
import fitz                      # PyMuPDF, reads PDFs
import numpy as np
import pandas as pd
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
!pip install pypdf sentence-transformers scikit-learn

import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FOLDER = '/content/drive/MyDrive/20_Job_Descriptions (2).pdf'

def read_pdf(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

In [43]:
JD_PDF_PATH   = "/content/drive/MyDrive/20_Job_Descriptions (2).pdf"  # the single PDF holding 20 JDs
RESUME_FOLDER = "/content/drive/MyDrive/Education"                   # folder with all resumes
TOP_K         = 10                                                # how many resumes per JD

EXPECTED_JDS  = 20
MODEL_NAME    = "all-MiniLM-L6-v2"   # fast & free. Use "all-mpnet-base-v2" for higher quality (slower)

In [33]:
def extract_pdf_pages(path):
    doc = fitz.open(path)
    pages = [page.get_text() for page in doc]
    doc.close()
    return pages

def load_job_descriptions(pdf_path, expected=20):
    pages = extract_pdf_pages(pdf_path)
    full_text = "\n".join(pages)

    # Strategy A: assume one job description per page
    by_page = [p.strip() for p in pages if p.strip()]
    if len(by_page) == expected:
        return by_page

    # Strategy B: split the whole text on common JD headings
    # -> tweak this regex to match how YOUR job descriptions start
    pattern = r"(?i)(?=job\s*description\s*\d*|job\s*title\s*[:\-]|position\s*[:\-]|role\s*[:\-])"
    by_pattern = [p.strip() for p in re.split(pattern, full_text) if p.strip()]
    if len(by_pattern) >= expected:
        return by_pattern[:expected]

    print(f"WARNING: page-split gave {len(by_page)}, pattern-split gave {len(by_pattern)}, "
          f"expected {expected}. Inspect the preview below and adjust the splitting logic.")
    return by_page if by_page else by_pattern

jd_texts = load_job_descriptions(JD_PDF_PATH, EXPECTED_JDS)
print(f"Loaded {len(jd_texts)} job descriptions\n")

# Preview the first line of each JD so you can confirm the split is correct
for i, t in enumerate(jd_texts, 1):
    first_line = t.strip().splitlines()[0] if t.strip() else "(empty)"
    print(f"{i:2d}. {first_line[:80]}")

Loaded 20 job descriptions

 1. 20 Professional Job Descriptions
 2. Azure DevOps Engineer
 3. Data Scientist
 4. Data Analyst
 5. Full Stack Developer
 6. Python Developer
 7. Machine Learning Engineer
 8. Generative AI Engineer
 9. Cloud Engineer
10. Site Reliability Engineer (SRE)
11. Business Analyst
12. SQL Developer
13. Power BI Developer
14. ETL Developer
15. Big Data Engineer
16. MLOps Engineer
17. Kubernetes Administrator
18. Cyber Security Analyst
19. Digital Marketing Specialist
20. Project Manager


In [35]:
def extract_resume_text(path):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".pdf":
            doc = fitz.open(path)
            text = "\n".join(page.get_text() for page in doc)
            doc.close()
            return text
        elif ext == ".docx":
            return "\n".join(p.text for p in Document(path).paragraphs)
        elif ext == ".txt":
            with open(path, "r", errors="ignore") as f:
                return f.read()
    except Exception as e:
        print(f"Could not read {path}: {e}")
    return ""

resume_paths = []
for ext in ("*.pdf", "*.docx", "*.txt"):
    resume_paths += glob.glob(os.path.join(RESUME_FOLDER, "**", ext), recursive=True)

resume_texts, resume_names = [], []
for p in sorted(resume_paths):
    txt = extract_resume_text(p)
    if txt.strip():
        resume_texts.append(txt)
        resume_names.append(os.path.basename(p))

print(f"Loaded {len(resume_texts)} resumes")

Loaded 5000 resumes


In [36]:
model = SentenceTransformer(MODEL_NAME)   # uses GPU automatically if the Colab runtime has one

jd_emb = model.encode(jd_texts, convert_to_numpy=True,
                      normalize_embeddings=True, show_progress_bar=True)
res_emb = model.encode(resume_texts, convert_to_numpy=True,
                       normalize_embeddings=True, show_progress_bar=True)

print("JD embeddings:", jd_emb.shape, " Resume embeddings:", res_emb.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

JD embeddings: (20, 384)  Resume embeddings: (5000, 384)


In [38]:
sim = cosine_similarity(jd_emb, res_emb)   # shape: (num_jds, num_resumes)

rows = []
k = min(TOP_K, len(resume_names))
for i in range(len(jd_texts)):
    top_idx = np.argsort(sim[i])[::-1][:k]
    for rank, idx in enumerate(top_idx, 1):
        rows.append({
            "JD_number": i + 1,
            "rank":      rank,
            "resume":    resume_names[idx],
            "score":     round(float(sim[i][idx]), 4),
        })

results = pd.DataFrame(rows)
results.head(TOP_K)

,JD_number,rank,resume,score
0,1,1,Resume_034049_Amanda_Walker.pdf,0.5929
1,1,2,Resume_033274_Shelly_Lewis.pdf,0.5913
2,1,3,Resume_030521_Suzanne_Bowman.pdf,0.5910
3,1,4,Resume_033028_Kaylee_Anderson.pdf,0.5898
4,1,5,Resume_033394_Steven_Watkins.pdf,0.5896
5,1,6,Resume_033674_Edgar_Watts.pdf,0.5867
6,1,7,Resume_032194_Cindy_Ramirez.pdf,0.5846
7,1,8,Resume_033927_Robert_Cline.pdf,0.5843
8,1,9,Resume_031631_Todd_Burgess.pdf,0.5717
9,1,10,Resume_033541_Mark_Zhang.pdf,0.5707


In [39]:
for i in range(len(jd_texts)):
    print(f"\n=== Job Description {i+1} ===")
    block = results[results.JD_number == i + 1]
    for _, r in block.iterrows():
        print(f"  {int(r['rank']):2d}. {r['resume']:<45} score={r['score']}")


=== Job Description 1 ===
   1. Resume_034049_Amanda_Walker.pdf               score=0.5929
   2. Resume_033274_Shelly_Lewis.pdf                score=0.5913
   3. Resume_030521_Suzanne_Bowman.pdf              score=0.591
   4. Resume_033028_Kaylee_Anderson.pdf             score=0.5898
   5. Resume_033394_Steven_Watkins.pdf              score=0.5896
   6. Resume_033674_Edgar_Watts.pdf                 score=0.5867
   7. Resume_032194_Cindy_Ramirez.pdf               score=0.5846
   8. Resume_033927_Robert_Cline.pdf                score=0.5843
   9. Resume_031631_Todd_Burgess.pdf                score=0.5717
  10. Resume_033541_Mark_Zhang.pdf                  score=0.5707

=== Job Description 2 ===
   1. Resume_033274_Shelly_Lewis.pdf                score=0.6417
   2. Resume_032481_Christopher_Smith.pdf           score=0.629
   3. Resume_034361_Brandon_Carter.pdf              score=0.613
   4. Resume_032436_Derrick_Diaz.pdf                score=0.6092
   5. Resume_032194_Cindy_Ramirez.pdf  

In [44]:
out_path = "/content/drive/MyDrive/csv/jd_resume_matches.csv"
results.to_csv(out_path, index=False)
print("Saved to", out_path)

from google.colab import files
files.download(out_path)

Saved to /content/drive/MyDrive/csv/jd_resume_matches.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>